# cMAB Simulation

This notebook shows a simulation framework for the contextual multi-armed bandit (cMAB). It allows to study the behaviour of the bandit algoritm, to evaluate results and to run experiments on simulated data under different context, reward and action settings.

In [1]:
import pandas as pd
from sklearn.datasets import make_classification

from pybandits.cmab import CmabBernoulli
from pybandits.cmab_simulator import CmabSimulator
from pybandits.model import BayesianLogisticRegression, StudentT

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.generics:GenericModel` has been moved to `pydantic.BaseModel`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


First we need to define the simulation parameters. The parameters are split into two parts. The general parameters contain:
- Number of update rounds
- Number of samples per batch of update round
- Seed for reproducibility
- Verbosity enabler
- Visualization enabler

The problem definition parameters contain:
- Number of groups
- Number of features

Data are processed in batches of size n>=1. Per each batch of simulated samples, the cMAB selects one action and collects the corresponding simulated reward for each sample. Then, prior parameters are updated based on returned rewards from recommended actions.

In [2]:
# general simulator parameters
n_updates = 10
batch_size = 100
random_seed = None
verbose = True
visualize = True

In [3]:
# problem definition simulation parameters
n_groups = 3
n_features = 5

Next, we initialize the context matrix $X$ and the groups of samples. Samples that belong to the same group have features that come from the same distribution.
Then, the action model and the cMAB are defined. We define three actions, each with a Bayesian Logistic Regression model. The model is defined by a Student-T prior for the intercept and a Student-T prior for each feature coefficient.

In [4]:
# init context matrix and groups
context, group = make_classification(
    n_samples=batch_size * n_updates, n_features=n_features, n_informative=n_features, n_redundant=0, n_classes=n_groups
)
group = [str(g) for g in group]

In [5]:
# define action model
actions = {
    "a1": BayesianLogisticRegression(
        alpha=StudentT(mu=1, sigma=2), betas=n_features * [StudentT()], update_method="VI"
    ),
    "a2": BayesianLogisticRegression(
        alpha=StudentT(mu=1, sigma=2), betas=n_features * [StudentT()], update_method="VI"
    ),
    "a3": BayesianLogisticRegression(
        alpha=StudentT(mu=1, sigma=2), betas=n_features * [StudentT()], update_method="VI"
    ),
}
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

Finally, we need to define the probabilities of positive rewards per each action/group, i.e. the ground truth ('Action A': 0.8 for group '0' means that if the bandits selects 'Action A' for samples that belong to group '0', then the environment will return a positive reward with 80% probability).


In [6]:
# init probability of rewards from the environment
probs_reward = pd.DataFrame(
    [[0.05, 0.80, 0.05], [0.80, 0.05, 0.05], [0.80, 0.05, 0.80]],
    columns=actions.keys(),
    index=[str(g) for g in range(n_groups)],
)
print("Probability of positive reward for each group/action:")
probs_reward

Probability of positive reward for each group/action:


,a1,a2,a3
0,0.05,0.80,0.05
1,0.80,0.05,0.05
2,0.80,0.05,0.80


Now, we initialize the cMAB as shown in the previous notebook and the CmabSimulator with the parameters set above.

In [7]:
# init simulation
cmab_simulator = CmabSimulator(
    mab=cmab,
    group=group,
    batch_size=batch_size,
    n_updates=n_updates,
    probs_reward=probs_reward,
    context=context,
    verbose=verbose,
)

/home/runner/work/pybandits/pybandits/pybandits/simulator.py:124: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  if not value.applymap(lambda x: 0 <= x <= 1).all().all():


Now, we can start simulation process by executing run() which performs the following steps:
```
For i=0 to n_updates:
    Extract batch[i] of samples from X
    Model recommends the best actions as the action with the highest reward probability to each simulated sample in batch[i] and collect corresponding simulated rewards
    Model priors are updated using information from recommended actions and returned rewards
```
Finally, we can visualize the results of the simulation. As defined in the ground truth: 'a2' was the action recommended the most for samples that belong to group '0', 'a1' to group '1' and both 'a1' and 'a3' to group '2'.

In [8]:
cmab_simulator.run()

/home/runner/work/pybandits/pybandits/pybandits/simulator.py:253: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self._results = pd.concat((self._results, batch_results), ignore_index=True)
/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/pymc/data.py:384: FutureWarning: Data is now always mutable. Specifying the `mutable` kwarg will raise an error in a future release
  warnings.warn(


/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/pytensor/link/c/cmodule.py:2959: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(


/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/rich/live.py:231: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Finished [100%]: Average Loss = 33.27


/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/pymc/data.py:384: FutureWarning: Data is now always mutable. Specifying the `mutable` kwarg will raise an error in a future release
  warnings.warn(


...

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/rich/live.py:231: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Finished [100%]: Average Loss = 12.073


2025-04-24 06:59:09.288 | INFO     | pybandits.simulator:_print_results:445 - Simulation results (first 10 observations):



2025-04-24 06:59:09.309 | INFO     | pybandits.simulator:_print_results:446 - Count of actions selected by the bandit: 



2025-04-24 06:59:09.312 | INFO     | pybandits.simulator:_print_results:447 - Observed proportion of positive rewards for each action:



Furthermore, we can examine the number of times each action was selected and the proportion of positive rewards for each action.

In [9]:
cmab_simulator.selected_actions_count

,action,a1,a2,a3,cum_a1,cum_a2,cum_a3
group,batch,,,,,,
0,0.0,14,5,8,14,5,8
1,0.0,7,12,18,7,12,18
2,0.0,16,11,9,16,11,9
0,1.0,27,3,4,41,8,12
1,1.0,23,6,3,30,18,21
2,1.0,25,6,3,41,17,12
0,2.0,18,5,13,59,13,25
1,2.0,27,5,7,57,23,28
2,2.0,12,1,12,53,18,24


In [10]:
cmab_simulator.positive_reward_proportion

proportion
action group           
a1     0       0.038462
       1       0.820106
       2       0.781095
a2     0       0.782609
       1       0.027027
       2       0.066667
a3     0       0.038961
       1       0.057971
       2       0.881579